In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from pydantic import BaseModel,Field

load_dotenv()

llm=ChatOpenAI(model="deepseek-chat",
               base_url=os.getenv("DEEPSEEK_BASE_URL"),
               api_key=os.getenv("DEEPSEEK_API_KEY"),
               temperature=0)

In [ ]:
from typing import Annotated, TypedDict
import uuid
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage
from langgraph.graph import START, MessagesState, StateGraph, add_messages,END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langchain_core.runnables import RunnableConfig

memory=MemorySaver()
in_memory_store=InMemoryStore()

class AgentState(TypedDict):
    messages:Annotated[list,add_messages]
    
def call_model(state:MessagesState,config:RunnableConfig,*,store:BaseStore):
    
    user_id=config['configurable']['user_id']
    
    namespace=('memories',user_id)
    memories = store.search(namespace)
    
    info="\n".join([d.value['data'] for d in memories])
    
    last_message=state['messages'][-1]
    
    store.put(namespace,str(uuid.uuid4()),{'data':last_message.content})
        
    system_prompt=f"Answer the user's question in context: {info}"
    result=llm.invoke([SystemMessage(content=system_prompt)]+state['messages'])
    
    store.put(namespace,str(uuid.uuid4()),{'data':result.content})
    return {'messages':[result]}


builder=StateGraph(AgentState)
builder.add_node('call_model',call_model)

builder.add_edge(START,'call_model')
builder.add_edge('call_model',END)

graph=builder.compile(checkpointer=memory,store=in_memory_store)

In [ ]:
from IPython.display import display, Image


display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config={'configurable':{'thread_id':"1"},'user_id':'1'}

async for event in graph.astream_events({'messages':["你好，介绍一下你自己"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

In [6]:
config={'configurable':{'thread_id':"1"},'user_id':'1'}

async for event in graph.astream_events({'messages':["你好，我是西瓜老师，介绍一下你自己"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

|哇|，|西瓜|老师|您好|！|🍉| |很高兴|认识|您|！

|我是|Deep|Se|ek|，|由|深度|求|索|公司|开发的|AI|助手|。|简单|来说|，|我就是|您的|智能|小|帮手|，|随时|待|命|！

|**|关于|我的|几个|亮点|：|**

|✨| **|完全|免费|**| -| |无论|您|问|多少|问题|，|我都|不会|收费|，|放心|使用|！

|📚| **|超|长|记忆|**| -| |我的|上下文|处理|能力|达到|1|M|，|可以|一口气|读完|《|三|体|》|三部|曲|那么多|内容|，|所以|您|给我|再|长的|资料|我也|能|消化|！

|🌐| **|联网|搜索|**| -| |我的|知识|截止|到|202|5|年|5|月|，|如果您|需要|更新的|资讯|，|手动|打开|联网|搜索|功能|，|我|就能|帮|您|查找|最新|信息|！

|📎| **|文件|处理|**| -| |支持|上传|图片|、|PDF|、|Word|、|Excel|、|PPT|等|文件|，|我会|读取|其中的|文字|内容|帮|您|分析|处理|

|🗣|️| **|多|端|使用|**| -| |网页|版|、|App|都可以|用|，|App|还|支持|语音|输入|，|特别|方便|！

|🎯| **|擅长|领域|**| -| |写作|、|编程|、|翻译|、|学习|辅导|、|数据分析|、|创意|策划|……|基本上|您|能|想到|的|，|我都|乐意|尝试|！

|西瓜|老师|，|您|今天|有什么|需要|我|帮忙|的吗|？|无论是|备课|资料|、|教学|创意|，|还是|其他|任何|问题|，|尽管|吩咐|！|我会|用|最|热情|、|最|细致|的态度|为您|服务|！|💪|😄|||

In [7]:
config={'configurable':{'thread_id':"1"},'user_id':'1'}

async for event in graph.astream_events({'messages':["你好 我是谁"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

|你好|呀|！|从|咱们|刚才|的|对话|来看|，|您是|**|西瓜|老师|**|呀|！|🍉| |很高兴|再次|见到|您|！

|不过|，|如果您|想|让我|更|了解|您|，|比如|您的|教学|领域|、|兴趣爱好|，|或者|有什么|特别|的需求|，|都可以|告诉我|哦|！|这样|我|就能|更好地|为您|提供|帮助|啦|～

|有什么|我可以|为您|效|劳|的吗|？|😊|||

In [8]:
config={'configurable':{'thread_id':"12"},'user_id':'1'}

async for event in graph.astream_events({'messages':["你好 我是谁"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

|从|咱们|刚才|的|对话|来看|，|您是|**|西瓜|老师|**|呀|！|🍉| |很高兴|再次|见到|您|！

|如果您|想|让我|更|了解|您|，|比如|您的|教学|领域|、|兴趣爱好|，|或者|有什么|特别|的需求|，|都可以|告诉我|哦|！|这样|我|就能|更好地|为您|提供|帮助|啦|～

|有什么|我可以|为您|效|劳|的吗|？|😊|||

In [9]:
config={'configurable':{'thread_id':"12"},'user_id':'3'}

async for event in graph.astream_events({'messages':["你好 我是谁"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

|哈哈|，|看来|您|想|考|考|我|呀|！|😄| |从|咱们|的|对话|记录|来看|，|您|就是|**|西瓜|老师|**|本人|没错|啦|～|（|如果|这是|您|第一次|和我|聊天|，|那|您|就是|一位|新|朋友|，|欢迎|欢迎|！|）

|不过|，|如果您|想|让我|更|准确地|“|认识|”|您|，|可以|悄悄|告诉我|您的|名字|、|身份|，|或者|您|希望|我怎么|称呼|您|？|这样|我|就能|更|贴|心地|为您|服务|啦|！|✨|

|有什么|需要|帮忙|的|，|尽管|说|哦|～|||

In [11]:
config={'configurable':{'thread_id':"14"},'user_id':'4'}

async for event in graph.astream_events({'messages':["你好 我是谁"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

|你好|！|😊|

|从|我们|目前的|对话|来看|，|我还|不知道|你的|具体|身份|呢|。|你|可能是|：

|-| |一位|对|AI|技术|感兴趣|的朋友|
|-| |正在|寻找|帮助|的|咨询|者|
|-| |或者|只是|随便|聊聊|的|网友|

|如果你|愿意|告诉我|更多|关于|你自己的|信息|，|比如|你的|名字|、|职业|、|兴趣|，|或者|你今天|想|聊|什么|话题|，|我会|很高兴|更好地|了解|你|，|并|为你|提供|更|贴|心的|帮助|！

|那么|，|你想|让我|怎么|称呼|你呢|？|或者|有什么|我可以|帮|你的|吗|？|😊|||

In [12]:
config={'configurable':{'thread_id':"145"},'user_id':'3'}

async for event in graph.astream_events({'messages':["你好 我是谁"]},config,stream_mode="values"):
    if event['event']=='on_chat_model_stream':
        print(event['data']['chunk'].content,end='|',flush=True)

|哈哈|，|您|又来|考|我|啦|！|😄| |从|咱们|的|对话|记录|来看|，|您|就是|**|西瓜|老师|**|本人|没错|啦|～|（|如果|这是|您|第一次|和我|聊天|，|那|您|就是|一位|新|朋友|，|欢迎|欢迎|！|）

|不过|，|如果您|想|让我|更|准确地|“|认识|”|您|，|可以|悄悄|告诉我|您的|名字|、|身份|，|或者|您|希望|我怎么|称呼|您|？|这样|我|就能|更|贴|心地|为您|服务|啦|！|✨|

|有什么|需要|帮忙|的|，|尽管|说|哦|～|||